# Advent of Code - Day 8

Find the Day 8 problem [here](https://adventofcode.com/2025/day/8").

## Pre-requisites: specify import(s), set constants, and load data.

In [ ]:
import math

PUZZLE_FILENAME = "data/day8.csv"
TEXT_ENCODING = "utf8"

In [ ]:
def load_data() -> list[tuple[int]]:
    """Loads the data from the day 8 file.
    
    Extract from data file:
    53058,31171,52130
    16732,38579,92679
    85580,58662,14262
    72245,188,83718
    14212,44745,26379
    ...
    
    Returns:
        A list of tuples where each tuple has three dimensions
        represnting a coordinate: (X, Y, Z)
    """
    with open(PUZZLE_FILENAME, mode="r", encoding=TEXT_ENCODING) as file:
        xyz = [tuple(map(int, line.rstrip().split(','))) for line in file]
    return xyz


In [ ]:
class UnionFind:
    """A Union-Find data structure for tracking connected nodes."""

    def __init__(self, n):
        self.parent = list(range(n))
        # All nodes start in circuit of size 1.
        self.size = [1] * n
    
    def find(self, x) -> int:
        """Find the root of the set containing x.
        
        If the root of the node is not itself, recurse until you find the
        node that does have itself as the root. Path compression.
        
        Args: x the value for which to find the root.
        """
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  
        return self.parent[x]
    
    def union(self, x, y) -> bool:
        """Join the two sets: x and y.
        
        Args:
            x, y the two sets to be joined.

        Returns:
            True if the two sets could be joined, or False if they are already joined.
        """
        px, py = self.find(x), self.find(y)
        if px == py:
            return False
    
        if self.size[px] < self.size[py]:
            px, py = py, px
        self.parent[py] = px
        self.size[px] += self.size[py]
        return True
    
    def get_size(self, x) -> int:
        """Returns the size of the supplied set."""
        return self.size[self.find(x)]
    
    def get_prod_three_largest_sizes(self) -> int:
        """Returns the product of the sizes of the three largest sets."""
        sizes = [self.size[i] for i in range(len(self.parent)) if self.parent[i] == i]
        sizes.sort(reverse=True)
        return math.prod(sizes[:3])


# Part One

### Notes

* Union find.

In [ ]:
def connect_closest_junction_boxes():
    coords = load_data()
    uf = UnionFind(len(coords))

    pairs = []
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):  # avoid duplicates
            dist = math.dist(coords[i], coords[j])
            pairs.append((dist, i, j))

    pairs.sort()

    # Connect the 1000 shortest pairs
    for k in range(1000):
        dist, i, j = pairs[k]
        uf.union(i, j)

    return uf.get_prod_three_largest_sizes()

In [ ]:
print(f"{connect_closest_junction_boxes()}")

# Part Two

### Notes

* Continue connecting the closest unconnected pairs of junction boxes together until they're all in the same circuit. What do you get if you multiply together the X coordinates of the last two junction boxes you need to connect?

In [ ]:
def create_one_circut_and_get_product_final_two_X():
    """Create one circuit (minimum connector) and provide product of final two X."""
    
    coords = load_data()
    uf = UnionFind(len(coords))

    pairs = []
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dist = math.dist(coords[i], coords[j])
            pairs.append((dist, i, j))

    pairs.sort()

    last_i, last_j = None, None

    for dist, i, j in pairs:
        if uf.union(i, j):
            last_i, last_j = i, j
            
            if uf.get_size(i) == len(coords):
                break

    return coords[last_i][0] * coords[last_j][0]

In [ ]:
print(f"Solution to part two: {create_one_circut_and_get_product_final_two_X()}")